**Generative AI Use**: For the purposes of the assignments, the use of generative AI is subject to the same policies regarding collaboration. Just as with other collaborators, each student must write down the solutions independently of the output of the interaction and the submission should include a note denoting the nature of the collaboration. The use of generative AI tools to substantially complete sections of the assignments is not in line with the spirit of the assignments, and would be a violation of the [Honor Code](https://communitystandards.stanford.edu/policies-and-guidance/honor-code).

---

**中文对照（参考译文，不含解答）**

**生成式 AI 的使用**：本作业对生成式 AI 的使用采用与协作相同的规则。与其他协作者交流之后，每位学生仍须独立写出自己的解答，不能直接依赖交流中得到的输出；提交时应附上说明，如实记录协作的性质。使用生成式 AI 实质性地完成作业的大部分内容，不符合本作业的教学目的，也会违反[学术诚信准则](https://communitystandards.stanford.edu/policies-and-guidance/honor-code)。

In [ ]:
# CS231n local environment setup (exercise code unchanged)
# [中文] CS231n 本地环境设置 (exercise code unchanged)
from pathlib import Path
import os
import sys

_candidates = [Path.cwd(), *Path.cwd().parents]
_workspace = next((p for p in _candidates if (p / "requirements-local.txt").is_file()
                   and (p / "assignment1" / "cs231n").is_dir()), None)
if _workspace is None:
    raise RuntimeError("Launch Jupyter from the CS231n workspace root.")
ASSIGNMENT_DIR = _workspace / "assignment1"
os.chdir(ASSIGNMENT_DIR)
if str(ASSIGNMENT_DIR) not in sys.path:
    sys.path.insert(0, str(ASSIGNMENT_DIR))
print("Python:", sys.executable)
print("Working directory:", Path.cwd())


# Softmax Classifier exercise

*Complete and hand in this completed worksheet (including its outputs and any supporting code outside of the worksheet) with your assignment submission. For more details see the [assignments page](http://vision.stanford.edu/teaching/cs231n/assignments.html) on the course website.*

In this exercise you will:
    
- implement a fully-vectorized **loss function** for the Softmax classifier.
- implement the fully-vectorized expression for its **analytic gradient**
- **check your implementation** using numerical gradient
- use a validation set to **tune the learning rate and regularization** strength
- **optimize** the loss function with **SGD**
- **visualize** the final learned weights


---

**中文对照（参考译文，不含解答）**

# 软体分类练习

* 完成并提交已完成的工作表(including its outputs and any supporting code outside of the worksheet)。 详情见课程网站上的 [assignments page](http://vision.stanford.edu/teaching/cs231n/assignments.html)。 *

在此练习中,您将:

- 为 Softmax 分类器实施完全矢量的 ** loss 函数。
- 实施其**分析梯度的完全矢量表达式**
- ** 使用数字梯度检查您的执行**
- 使用验证集** 显示学习率和正则化** 强度
- ** 以** SGD优化** 损失函数
- ** 可视化**最后的学习权重。


In [ ]:
# Run some setup code for this notebook.
# [中文] 运行这个笔记本的设置代码。
import random
import numpy as np
from cs231n.data_utils import load_CIFAR10
import matplotlib.pyplot as plt

# This is a bit of magic to make matplotlib figures appear inline in the
# notebook rather than in a new window.
# [中文] 这有点神奇,让 Mattlotlib 数字在笔记本中出现,而不是在新窗口中出现。
%matplotlib inline
plt.rcParams['figure.figsize'] = (10.0, 8.0) # set default size of plots
# [中文] 设置默认的绘图大小
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'

# Some more magic so that the notebook will reload external python modules;
# see http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
# [中文] 使笔记本能重新装入外部 python 模块的更多魔术;
# 见http://stackoverflow.com/questions/1907993/autoreload-of-modules-in-ipython
%load_ext autoreload
%autoreload 2

## CIFAR-10 Data Loading and Preprocessing

---

**中文对照（参考译文，不含解答）**

## CIFAR - 10 数据装入和预处理

In [ ]:
# Load the raw CIFAR-10 data.
# [中文] 装入原始的CIFAR - 10 数据。
cifar10_dir = 'cs231n/datasets/cifar-10-batches-py'

# Cleaning up variables to prevent loading data multiple times (which may cause memory issue)
# [中文] 清理变量以防止多次装入数据 (which may cause memory issue)
try:
   del X_train, y_train
   del X_test, y_test
   print('Clear previously loaded data.')
except:
   pass

X_train, y_train, X_test, y_test = load_CIFAR10(cifar10_dir)

# As a sanity check, we print out the size of the training and test data.
# [中文] 作为精神检查,我们打印出 训练和测试数据的大小。
print('Training data shape: ', X_train.shape)
print('Training labels shape: ', y_train.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

In [ ]:
# Visualize some examples from the dataset.
# We show a few examples of training images from each class.
# [中文] 从数据集中可视化一些实例。 我们展示了每个班级的一些训练图像实例。
classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
num_classes = len(classes)
samples_per_class = 7
for y, cls in enumerate(classes):
    idxs = np.flatnonzero(y_train == y)
    idxs = np.random.choice(idxs, samples_per_class, replace=False)
    for i, idx in enumerate(idxs):
        plt_idx = i * num_classes + y + 1
        plt.subplot(samples_per_class, num_classes, plt_idx)
        plt.imshow(X_train[idx].astype('uint8'))
        plt.axis('off')
        if i == 0:
            plt.title(cls)
plt.show()

In [ ]:
# Split the data into train, val, and test sets. In addition we will
# create a small development set as a subset of the training data;
# we can use this for development so our code runs faster.
# [中文] 将数据分成火车、 val 和 测试组。 此外, 我们将创建一个小开发组作为训练数据的一个子集; 我们可以用它来开发, 这样我们的代码就能运行得更快。
num_training = 49000
num_validation = 1000
num_test = 1000
num_dev = 500

# Our validation set will be num_validation points from the original
# training set.
# [中文] 我们的验证集将是原始训练集的num_validation点。
mask = range(num_training, num_training + num_validation)
X_val = X_train[mask]
y_val = y_train[mask]

# Our training set will be the first num_train points from the original
# training set.
# [中文] 我们的训练组将是最初训练组的第一个num_train点。
mask = range(num_training)
X_train = X_train[mask]
y_train = y_train[mask]

# We will also make a development set, which is a small subset of
# the training set.
# [中文] 我们还将制作一套发展教材,这是一套训练教材的一小部分。
mask = np.random.choice(num_training, num_dev, replace=False)
X_dev = X_train[mask]
y_dev = y_train[mask]

# We use the first num_test points of the original test set as our
# test set.
# [中文] 我们使用原始测试组的第一个num_test点作为测试组。
mask = range(num_test)
X_test = X_test[mask]
y_test = y_test[mask]

print('Train data shape: ', X_train.shape)
print('Train labels shape: ', y_train.shape)
print('Validation data shape: ', X_val.shape)
print('Validation labels shape: ', y_val.shape)
print('Test data shape: ', X_test.shape)
print('Test labels shape: ', y_test.shape)

In [ ]:
# Preprocessing: reshape the image data into rows
# [中文] 预处理: 将图像数据重塑为行
X_train = np.reshape(X_train, (X_train.shape[0], -1))
X_val = np.reshape(X_val, (X_val.shape[0], -1))
X_test = np.reshape(X_test, (X_test.shape[0], -1))
X_dev = np.reshape(X_dev, (X_dev.shape[0], -1))

# As a sanity check, print out the shapes of the data
# [中文] 作为理智检查,打印数据形状
print('Training data shape: ', X_train.shape)
print('Validation data shape: ', X_val.shape)
print('Test data shape: ', X_test.shape)
print('dev data shape: ', X_dev.shape)

In [ ]:
# Preprocessing: subtract the mean image
# first: compute the image mean based on the training data
# [中文] 预处理: 先减去平均图像: 根据训练数据计算图像平均值
mean_image = np.mean(X_train, axis=0)
print(mean_image[:10]) # print a few of the elements
# [中文] 打印一些元素
plt.figure(figsize=(4,4))
plt.imshow(mean_image.reshape((32,32,3)).astype('uint8')) # visualize the mean image
# [中文] 视觉化平均图像
plt.show()

# second: subtract the mean image from train and test data
# [中文] 第二:从列车和测试数据中减去平均图像
X_train -= mean_image
X_val -= mean_image
X_test -= mean_image
X_dev -= mean_image

# third: append the bias dimension of ones (i.e. bias trick) so that our classifier
# only has to worry about optimizing a single weight matrix W.
# [中文] 第三:附加偏差尺寸( i.e.偏差技巧),这样我们的分类者只需担心优化一个单一的权重矩阵 W 即可。
X_train = np.hstack([X_train, np.ones((X_train.shape[0], 1))])
X_val = np.hstack([X_val, np.ones((X_val.shape[0], 1))])
X_test = np.hstack([X_test, np.ones((X_test.shape[0], 1))])
X_dev = np.hstack([X_dev, np.ones((X_dev.shape[0], 1))])

print(X_train.shape, X_val.shape, X_test.shape, X_dev.shape)

## Softmax Classifier

Your code for this section will all be written inside `cs231n/classifiers/softmax.py`.

As you can see, we have prefilled the function `softmax_loss_naive` which uses for loops to evaluate the softmax loss function.

---

**中文对照（参考译文，不含解答）**

## 软体分类器

您对本节的代码将全部写入`cs231n/classifiers/softmax.py` 内。

如您所见,我们已经预填了用于循环的 `softmax_loss_naive` 函数,用于评估软负损失函数。

In [ ]:
# Evaluate the naive implementation of the loss we provided for you:
# [中文] 评估我们为你们提供的损失的朴素实现情况:
from cs231n.classifiers.softmax import softmax_loss_naive
import time

# generate a random Softmax classifier weight matrix of small numbers
# [中文] 生成一个随机的 Softmax 分类器小数字的权重矩阵
W = np.random.randn(3073, 10) * 0.0001

loss, grad = softmax_loss_naive(W, X_dev, y_dev, 0.000005)
print('loss: %f' % (loss, ))

# As a rough sanity check, our loss should be something close to -log(0.1).
# [中文] 我们的损失应该是接近 -log (0.1)。
print('loss: %f' % loss)
print('sanity check: %f' % (-np.log(0.1)))

**Inline Question 1**

Why do we expect our loss to be close to -log(0.1)? Explain briefly.**

$\color{blue}{\textit Your Answer:}$ *Fill this in*



---

**中文对照（参考译文，不含解答）**

**随堂问题 1**

为什么预期初始损失接近 -log(0.1)？请简要解释。

**你的回答：** [请自行填写]

The `grad` returned from the function above is right now all zero. Derive and implement the gradient for the softmax loss function and implement it inline inside the function `softmax_loss_naive`. You will find it helpful to interleave your new code inside the existing function.

To check that you have correctly implemented the gradient, you can numerically estimate the gradient of the loss function and compare the numeric estimate to the gradient that you computed. We have provided code that does this for you:

---

**中文对照（参考译文，不含解答）**

从上方函数返回的 `grad` 现在全部为零。 在函数 `softmax_loss_naive` 中, 提取并安装软负损失函数的梯度, 并内嵌它。 您会发现将您的新代码插入到现有函数中有帮助。

要检查您是否正确实施了梯度, 您可以用数字估计损失函数的梯度, 并将数字估计与您计算的梯度进行比较。 我们为您提供了代码:

In [ ]:
# Once you've implemented the gradient, recompute it with the code below
# and gradient check it with the function we provided for you
# [中文] 一旦你应用了梯度, 用下面的代码重新计算, 并用我们提供的函数检查它

# Compute the loss and its gradient at W.
# [中文] 计算损失及其在W的梯度。
loss, grad = softmax_loss_naive(W, X_dev, y_dev, 0.0)

# Numerically compute the gradient along several randomly chosen dimensions, and
# compare them with your analytically computed gradient. The numbers should match
# almost exactly along all dimensions.
# [中文] 按几个随机选择的维度计算梯度,并将其与分析计算梯度进行比较。数字应该与所有维度几乎完全吻合。
from cs231n.gradient_check import grad_check_sparse
f = lambda w: softmax_loss_naive(w, X_dev, y_dev, 0.0)[0]
grad_numerical = grad_check_sparse(f, W, grad)

# do the gradient check once again with regularization turned on
# you didn't forget the regularization gradient did you?
# [中文] 您是否再次检查梯度, 并打开了正则化, 您没有忘记正则化梯度吗 ?
loss, grad = softmax_loss_naive(W, X_dev, y_dev, 5e1)
f = lambda w: softmax_loss_naive(w, X_dev, y_dev, 5e1)[0]
grad_numerical = grad_check_sparse(f, W, grad)

**Inline Question 2**

Although gradcheck is reliable softmax loss, it is possible that for SVM loss, once in a while, a dimension in the gradcheck will not match exactly. What could such a discrepancy be caused by? Is it a reason for concern? What is a simple example in one dimension where a svm loss gradient check could fail? How would change the margin affect of the frequency of this happening?

Note that SVM loss for a sample $(x_i, y_i)$ is defined as: $$L_i = \sum_{j\ne y_i}\max(0, s_j - s_{y_i} + \Delta)$$ where $j$ iterates over all classes except the correct class $y_i$ and $s_j$ denotes the classifier score for $j^{th}$ class. $\Delta$ is a scalar margin. For more information, refer to 'Multiclass Support Vector Machine loss' on [this](https://cs231n.github.io/linear-classify/) page.

*Hint: the SVM loss function is not strictly speaking differentiable.*


$\color{blue}{\textit Your Answer:}$ *fill this in.*  



---

**中文对照（参考译文，不含解答）**

**随堂问题 2**

数值梯度检查对 Softmax 损失通常可靠，但对于 SVM 损失，偶尔会在某个维度上出现数值梯度与解析梯度无法完全匹配的情况。什么原因可能导致这种差异？是否需要担心？能否给出一个一维例子，说明 SVM 损失的梯度检查何时会失败？改变间隔参数会怎样影响这种情况发生的频率？

样本 $(x_i, y_i)$ 的 SVM 损失定义为：
$$L_i = \sum_{j\ne y_i}\max(0, s_j - s_{y_i} + \Delta)$$
其中，$j$ 遍历正确类别 $y_i$ 以外的所有类别，$s_j$ 表示分类器对第 $j^{th}$ 个类别的评分，$\Delta$ 是标量间隔参数。详情见[课程页面](https://cs231n.github.io/linear-classify/)中的“多类别支持向量机损失”。

*原题提示：严格来说，SVM 损失函数并非处处可微。*

**你的回答：** [请自行填写]

In [ ]:
# Next implement the function softmax_loss_vectorized; for now only compute the loss;
# we will implement the gradient in a moment.
# [中文] 下一步执行函数 softmax_loss_vectorized; 现在只计算损失; 我们稍后会执行梯度。
tic = time.time()
loss_naive, grad_naive = softmax_loss_naive(W, X_dev, y_dev, 0.000005)
toc = time.time()
print('Naive loss: %e computed in %fs' % (loss_naive, toc - tic))

from cs231n.classifiers.softmax import softmax_loss_vectorized
tic = time.time()
loss_vectorized, _ = softmax_loss_vectorized(W, X_dev, y_dev, 0.000005)
toc = time.time()
print('Vectorized loss: %e computed in %fs' % (loss_vectorized, toc - tic))

# The losses should match but your vectorized implementation should be much faster.
# [中文] 损失应该与损失相符,但你的矢量化实施应该更快。
print('difference: %f' % (loss_naive - loss_vectorized))

In [ ]:
# Complete the implementation of softmax_loss_vectorized, and compute the gradient
# of the loss function in a vectorized way.
# [中文] 完成 softmax_loss_vectorized 的安装,并以矢量化的方式计算损失函数的梯度。

# The naive implementation and the vectorized implementation should match, but
# the vectorized version should still be much faster.
# [中文] 幼稚的执行和矢量化的执行应当吻合,但矢量化的版本仍应快得多。
tic = time.time()
_, grad_naive = softmax_loss_naive(W, X_dev, y_dev, 0.000005)
toc = time.time()
print('Naive loss and gradient: computed in %fs' % (toc - tic))

tic = time.time()
_, grad_vectorized = softmax_loss_vectorized(W, X_dev, y_dev, 0.000005)
toc = time.time()
print('Vectorized loss and gradient: computed in %fs' % (toc - tic))

# The loss is a single number, so it is easy to compare the values computed
# by the two implementations. The gradient on the other hand is a matrix, so
# we use the Frobenius norm to compare them.
# [中文] 损失是一个数字, 所以比较两个执行过程所计算的值很容易。 而梯度是一个矩阵, 所以我们使用 Frobenius 规范来比较它们。
difference = np.linalg.norm(grad_naive - grad_vectorized, ord='fro')
print('difference: %f' % difference)

### Stochastic Gradient Descent

We now have vectorized and efficient expressions for the loss, the gradient and our gradient matches the numerical gradient. We are therefore ready to do SGD to minimize the loss. Your code for this part will be written inside `cs231n/classifiers/linear_classifier.py`.

---

**中文对照（参考译文，不含解答）**

### 随机梯度下降

我们现在对损失、 梯度 和 我们的梯度 进行了矢量化和高效表达。 因此, 我们准备做 SGD 来最小化损失。 您对这部分的代码将写入 `cs231n/classifiers/linear_classifier.py` 内。

In [ ]:
# In the file linear_classifier.py, implement SGD in the function
# LinearClassifier.train() and then run it with the code below.
# [中文] 在文件linear_classifier.py中,在函数LinearClassifier.train()中执行 SGD,然后用下面的代码运行。
from cs231n.classifiers import Softmax
softmax = Softmax()
tic = time.time()
loss_hist = softmax.train(X_train, y_train, learning_rate=1e-7, reg=2.5e4,
                      num_iters=1500, verbose=True)
toc = time.time()
print('That took %fs' % (toc - tic))

In [ ]:
# A useful debugging strategy is to plot the loss as a function of
# iteration number:
# [中文] 一个有用的调试策略是将损失作为迭代编号的函数进行绘图:
plt.plot(loss_hist)
plt.xlabel('Iteration number')
plt.ylabel('Loss value')
plt.show()

In [ ]:
# Write the LinearClassifier.predict function and evaluate the performance on
# both the training and validation set
# You should get validation accuracy of about 0.34 (> 0.33).
# [中文] 写入 LinearClassifier.predict 函数, 并在训练和验证组上评估性能, 您应该获得大约 0.34 ( > 0.33 ) 的验证准确性
# 。
y_train_pred = softmax.predict(X_train)
print('training accuracy: %f' % (np.mean(y_train == y_train_pred), ))
y_val_pred = softmax.predict(X_val)
print('validation accuracy: %f' % (np.mean(y_val == y_val_pred), ))

In [ ]:
# Save the trained model for autograder.
# [中文] 将经过训练的模型保存到自动升级器中。
softmax.save("softmax.npy")

In [ ]:
# Use the validation set to tune hyperparameters (regularization strength and
# learning rate). You should experiment with different ranges for the learning
# rates and regularization strengths; if you are careful you should be able to
# get a classification accuracy of about 0.365 (> 0.36) on the validation set.
# [中文] 使用验证集来调和超参数 (regularization strength and learning rate)。 您应该试验不同的学习率和正则化强度范围;
# 如果您小心, 您应该能够在验证集上获得大约 0.365 ( > 0.36 ) 的分类准确率。

# Note: you may see runtime/overflow warnings during hyper-parameter search.
# This may be caused by extreme values, and is not a bug.
# [中文] 注意: 您在超参数搜索时可能会看到运行时间/ 过流警告。 这可能是由极端值造成的, 而不是错误。

# results is dictionary mapping tuples of the form
# (learning_rate, regularization_strength) to tuples of the form
# (training_accuracy, validation_accuracy). The accuracy is simply the fraction
# of data points that are correctly classified.
# [中文] 结果为窗体(learning_rate, regularization_strength) 的字典图示图例,以窗体 (training_accuracy,
# validation_accuracy) 的图示图示。精确度只是被正确分类的数据点的分数。
results = {}
best_val = -1   # The highest validation accuracy that we have seen so far.
# [中文] 这是迄今为止我们所看到的最高的验证准确性。
best_softmax = None # The Softmax object that achieved the highest validation rate.
# [中文] 达到最高验证率的软体物体。

################################################################################
# TODO:                                                                        #
# Write code that chooses the best hyperparameters by tuning on the validation #
# set. For each combination of hyperparameters, train a Softmax on the.        #
# training set, compute its accuracy on the training and validation sets, and  #
# store these numbers in the results dictionary. In addition, store the best   #
# validation accuracy in best_val and the Softmax object that achieves this.   #
# accuracy in best_softmax.                                                    #
#                                                                              #
# Hint: You should use a small value for num_iters as you develop your         #
# validation code so that the classifiers don't take much time to train; once  #
# you are confident that your validation code works, you should rerun the      #
# code with a larger value for num_iters.                                      #
################################################################################
# [中文] TODO: 通过调试校准集来选择最佳超参数的代码。 对于每组合的超参数, 在训练集上训练一个 Softmax, 计算其训练和校准组的准确性,
# 并将这些数字存储在结果字典中。 此外, 将最好的校验准确率存储在best_val和达到此目标的软体对象。best_softmax. 提示:
# 您应该使用一个小值num_iters当您开发您的校验代码, 以便分类者不会花太多时间来训练; 一旦您确信您的校准代码有效, 您就应该重新运行代码,
# 而重新运行的价值更大。num_iters .

# Provided as a reference. You may or may not want to change these hyperparameters
# [中文] 提供作为参考。 您可能想要或不想更改这些超参数。
learning_rates = [1e-7, 1e-6]
regularization_strengths = [2.5e4, 1e4]



# Print out results.
# [中文] 打印结果。
for lr, reg in sorted(results):
    train_accuracy, val_accuracy = results[(lr, reg)]
    print('lr %e reg %e train accuracy: %f val accuracy: %f' % (
                lr, reg, train_accuracy, val_accuracy))

print('best validation accuracy achieved during cross-validation: %f' % best_val)

In [ ]:
# Visualize the cross-validation results
# [中文] 可视化交叉验证结果
import math
import pdb

# pdb.set_trace()
# [中文]  pdb.set_trace()

x_scatter = [math.log10(x[0]) for x in results]
y_scatter = [math.log10(x[1]) for x in results]

# plot training accuracy
# [中文] 绘图训练准确率
marker_size = 100
colors = [results[x][0] for x in results]
plt.subplot(2, 1, 1)
plt.tight_layout(pad=3)
plt.scatter(x_scatter, y_scatter, marker_size, c=colors, cmap=plt.cm.coolwarm)
plt.colorbar()
plt.xlabel('log learning rate')
plt.ylabel('log regularization strength')
plt.title('CIFAR-10 training accuracy')

# plot validation accuracy
# [中文] 绘图验证准确率
colors = [results[x][1] for x in results] # default size of markers is 20
# [中文] 标记的默认大小为 20
plt.subplot(2, 1, 2)
plt.scatter(x_scatter, y_scatter, marker_size, c=colors, cmap=plt.cm.coolwarm)
plt.colorbar()
plt.xlabel('log learning rate')
plt.ylabel('log regularization strength')
plt.title('CIFAR-10 validation accuracy')
plt.show()

In [ ]:
# Evaluate the best softmax on test set
# [中文] 评估测试集上最好的软键
y_test_pred = best_softmax.predict(X_test)
test_accuracy = np.mean(y_test == y_test_pred)
print('Softmax classifier on raw pixels final test set accuracy: %f' % test_accuracy)

In [ ]:
# Save best softmax model
# [中文] 保存最好的软体模型
best_softmax.save("best_softmax.npy")

In [ ]:
# Visualize the learned weights for each class.
# Depending on your choice of learning rate and regularization strength, these may
# or may not be nice to look at.
# [中文] 视觉化每个班级的学习权重。 取决于您对学习率和正则化强度的选择, 这些可能或可能不是很好看。
w = best_softmax.W[:-1,:] # strip out the bias
# [中文] 消除偏置
w = w.reshape(32, 32, 3, 10)
w_min, w_max = np.min(w), np.max(w)
classes = ['plane', 'car', 'bird', 'cat', 'deer', 'dog', 'frog', 'horse', 'ship', 'truck']
for i in range(10):
    plt.subplot(2, 5, i + 1)

    # Rescale the weights to be between 0 and 255
    # [中文] 调整权重在 0 和 255 之间
    wimg = 255.0 * (w[:, :, :, i].squeeze() - w_min) / (w_max - w_min)
    plt.imshow(wimg.astype('uint8'))
    plt.axis('off')
    plt.title(classes[i])

**Inline question 3**

Describe what your visualized Softmax classifier weights look like, and offer a brief explanation for why they look the way they do.

$\color{blue}{\textit Your Answer:}$ *fill this in*  


---

**中文对照（参考译文，不含解答）**

**随堂问题 3**

描述可视化后的 Softmax 分类器权重呈现什么样的图案，并简要解释为什么会呈现这样的图案。

**你的回答：** [请自行填写]

**Inline Question 4** - *True or False*

Suppose the overall training loss is defined as the sum of the per-datapoint loss over all training examples. It is possible to add a new datapoint to a training set that would change the softmax loss, but leave the SVM loss unchanged.

$\color{blue}{\textit Your Answer:}$


$\color{blue}{\textit Your Explanation:}$



---

**中文对照（参考译文，不含解答）**

**随堂问题 4——判断正误**

假设总体训练损失定义为所有训练样本逐样本损失之和。是否可能向训练集添加一个新样本，使 Softmax 损失发生变化，而 SVM 损失保持不变？

**你的回答：** [请自行填写]

**你的解释：** [请自行填写]